<a href="https://colab.research.google.com/github/alexcho2823-del/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item (one product within one order).

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd

lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
print(lines)

assert len(lines) == 4
assert lines['qty'].sum() == 7

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07


json_normalize creates one row for each entry inside every order’s lines list and copies that order’s order and vendor_id onto each row, which gives you 4 rows totaling 7 units.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors = pd.json_normalize(vendors_json)
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)
print(joined)

unmatched = joined[joined['_merge'] == 'left_only']
print("Unmatched vendor:", unmatched['vendor_id'].unique())
print("Units attached:", unmatched['qty'].sum())


           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only
Unmatched vendor: ['V-07']
Units attached: 3


The left join keeps every line item, even ones without a vendor, and the _merge column labels any row that found no match as left_only. Here that row is V-07 with 3 units.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# Decision: keep the unmatched vendor as "UNKNOWN (V-07)" instead of dropping it.
# Why: those 3 units really were sold, so dropping them would make the total show 4
# instead of 7, and the label flags the missing vendor record for follow-up.
joined['name'] = joined['name'].fillna('UNKNOWN (' + joined['vendor_id'] + ')')

per_vendor = joined.groupby('name')['qty'].sum().sort_values(ascending=False)
print(per_vendor)
assert per_vendor.sum() == 7

name
Hoos Burgers      3
UNKNOWN (V-07)    3
Cav Merch         1
Name: qty, dtype: int64


labeling the missing name keeps V-07’s units visible in the table. Without the label, groupby would silently drop the NaN name and the vendor totals would add up to only 4.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
per_zone = joined.groupby('zone')['qty'].sum()
print(per_zone)                      # Zone A: 4

print("Zone total:", per_zone.sum())                                     # 4
print("Units with no zone:", joined.loc[joined['zone'].isna(), 'qty'].sum())  # 3

print(joined.groupby('zone', dropna=False)['qty'].sum())


zone
A    4
Name: qty, dtype: int64
Zone total: 4
Units with no zone: 3
zone
A      4
NaN    3
Name: qty, dtype: int64


groupby drops NaN keys by default, so V-07’s 3 units vanish from the zone totals. Using dropna=False brings them back as a NaN row, which shows where the difference went. Zone A sold the most (4 units). Zone totals sum to 4, not 7, because V-07’s 3 units have no zone.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

safe = [{**o, 'lines': o.get('lines', [])} for o in ragged_json]
ragged = pd.json_normalize(safe, record_path='lines', meta=['order', 'vendor_id'])

ragged['qty'] = ragged['qty'].astype('Int64')
print(ragged)
print(ragged.dtypes)

no_lines = [o['order'] for o in ragged_json if 'lines' not in o]
print("Orders with no lines:", no_lines)   # [5]

    item   qty order vendor_id
0   Soda     2     4      V-01
1  Fries  <NA>     6      V-01
item         object
qty           Int64
order        object
vendor_id    object
dtype: object
Orders with no lines: [5]


Order 5 is missing its lines key, so .get('lines', []) gives it an empty list and it produces zero rows instead of a KeyError. Converting qty to Int64 makes the Fries row show 'NA' rather than 0.

### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

The quantity exists on each line item and not on the order. With one row per line item, questions like "total units per vendor/item/zone" would become a single gropuby sum. With one row per order the lines are a nested list so you wold ahve to loop through or explode it before counting anything. And you also cold not easily ask something like "how many sodas sold"

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

A quantity of zero is a known fact... nothing was sold. A missing quantity just means we do not know, due to bad or incomplete data. If we treat missing as 0, totals woukd then look exact when they are actually understated, averages get brought down, and the data problem will disappears. Nobody would know to go back and fix the Fries record. Keeping ''NA'' lets you count the unknowns, exclude them on purpose, or flag them.

_your answer here_